# BPATMP — Training trên Colab Pro (A100)

**Cách dùng:** Runtime → Change runtime type → **A100 GPU** → rồi **Runtime → Run all**.

Notebook tự động: clone repo → tải dataset HuggingFace → train (tự đánh giá **val** mỗi epoch, log W&B project `recsys-graph`) → đánh giá **test**.

> 🔑 **W&B:** thêm API key vào Colab Secrets (biểu tượng 🔑) tên `WANDB_API_KEY`. Nếu repo private, thêm `GITHUB_TOKEN`.

## 1. Kiểm tra GPU (phải là A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 2. Cài dependencies
Colab đã có sẵn torch+CUDA, pandas, pyarrow, scipy, pyyaml, numpy. Code chỉ cần thêm `torch_geometric` (không cần torch_scatter/sparse/cluster), `wandb`, `huggingface_hub`.

In [ ]:
!pip -q install torch_geometric wandb huggingface_hub hf_transfer
import torch_geometric, wandb, huggingface_hub
print('torch_geometric', torch_geometric.__version__, '| wandb', wandb.__version__, '| hf_hub', huggingface_hub.__version__)

## 3. Clone repo
Nếu repo **private**: thêm secret `GITHUB_TOKEN` rồi bỏ comment khối token bên dưới.

In [ ]:
%cd /content
REPO = 'https://github.com/nguyenmaiductrong/heterogeneous-graph-recsys.git'
# --- Repo private? Bỏ comment 3 dòng sau ---
# from google.colab import userdata
# TOK = userdata.get('GITHUB_TOKEN')
# REPO = f'https://{TOK}@github.com/nguyenmaiductrong/heterogeneous-graph-recsys.git'
import os
if not os.path.isdir('/content/heterogeneous-graph-recsys'):
    !git clone $REPO
%cd /content/heterogeneous-graph-recsys
!git log --oneline -1

## 4. Tải dataset HuggingFace → `/content/data`
[`nguyenmaiductrong/rees46-full-temporal`](https://huggingface.co/datasets/nguyenmaiductrong/rees46-full-temporal) — ~3.6 GB artifact đã tiền xử lý (feed thẳng vào training, không cần Spark).

In [ ]:
import os, time
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
t0 = time.time()
snapshot_download(repo_id='nguyenmaiductrong/rees46-full-temporal', repo_type='dataset',
                  local_dir='/content/data', max_workers=8)
print(f'Tải xong trong {time.time()-t0:.0f}s')
import json; print('node_counts:', json.load(open('/content/data/node_counts.json')))
!ls /content/data | head

## 5. Cấu hình W&B (project = `recsys-graph`) + đăng nhập

In [ ]:
import os
from pathlib import Path
import yaml

P = 'config/training.yaml'
cfg = yaml.safe_load(open(P))

WANDB_ENTITY = 'nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng'
cfg['wandb']['project'] = 'recsys-graph'
cfg['wandb']['entity'] = WANDB_ENTITY
cfg['wandb']['enabled'] = True
cfg['data']['data_dir'] = '/content/data/'
cfg['data']['struct_dir'] = '/content/data/node_mappings'
cfg['training']['progress_bar'] = False
cfg['training']['quiet_checkpoint_logs'] = True
os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = cfg['wandb']['project']
os.environ['WANDB_SILENT'] = 'true'
yaml.safe_dump(cfg, open(P, 'w'), sort_keys=False, allow_unicode=True)

# Checkpoint artifact ~1-1.5GB: tăng thời gian verify cloud và tránh artifact ref entity=None.
cm_path = Path('src/training/checkpoint_manager.py')
txt = cm_path.read_text()
txt = txt.replace('verify_timeout_secs: int = 300,', 'verify_timeout_secs: int = 3600,')
txt = txt.replace('verify_poll_secs: int = 30,', 'verify_poll_secs: int = 60,')
txt = txt.replace('wandb.Api(timeout=60).artifact', 'wandb.Api(timeout=180).artifact')
old = """        self._save_run_id(self.run.id)\n        logger.info(\"W&B run ready: id=%s\", self.run.id)\n"""
new = """        self.project = self.run.project or self.project\n        self.entity = self.run.entity or self.entity\n        if not self.entity:\n            raise RuntimeError(\n                \"W&B entity is empty after wandb.init(). Set wandb.entity in config.\"\n            )\n        self._save_run_id(self.run.id)\n        logger.info(\n            \"W&B run ready: entity=%s project=%s name=%s id=%s\",\n            self.entity, self.project, self.run.name, self.run.id,\n        )\n"""
if old in txt:
    txt = txt.replace(old, new)
txt = txt.replace(
    "    def _cleanup_old_checkpoints(self, keep: Path) -> None:\n        for pt in list(self.local_dir.glob(\"*.pt\")) + list(self.local_dir.glob(\"*.pth\")):\n            if pt.resolve() != keep.resolve():",
    "    def _cleanup_old_checkpoints(self, keep: Path) -> None:\n        for pt in list(self.local_dir.glob(\"*.pt\")) + list(self.local_dir.glob(\"*.pth\")):\n            if pt.name == \"best.pt\":\n                continue\n            if pt.resolve() != keep.resolve():",
)
cm_path.write_text(txt)

print('wandb.entity =', cfg['wandb']['entity'], '| wandb.project =', cfg['wandb']['project'], '| data_dir =', cfg['data']['data_dir'])
print('checkpoint verify_timeout_secs = 3600 | verify_poll_secs = 60 | wandb.Api timeout = 180')
print('colab logs: progress_bar =', cfg['training']['progress_bar'], '| quiet_checkpoint_logs =', cfg['training']['quiet_checkpoint_logs'])
print('model:', cfg['model'])
print('training: epochs=%s batch=%s eval_every=%s eval_subsample=%s device=%s save_dir=%s' % (
    cfg['training']['epochs'], cfg['training']['batch_size'], cfg['training'].get('eval_every'),
    cfg['training'].get('eval_subsample'), cfg['training']['device'], cfg['training']['save_dir']))

import wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
except Exception as e:
    print('Không thấy secret WANDB_API_KEY, đăng nhập thủ công:', e)
    wandb.login()

## 6. Training (đã bao gồm tự đánh giá VAL)
`scripts/run_training.py` tự động **mỗi epoch**: chạy `eval_epoch` trên **val**, log HR/NDCG@{1,5,10,20,50} lên W&B, chọn `best.pt` theo **NDCG@20**, early stopping (`patience`). Kết thúc còn chạy **full-val trên toàn bộ user** với `best.pt`.

40 epoch trên toàn bộ dữ liệu (983K user / 100K item) — tùy A100 có thể mất vài giờ. Muốn thử nhanh: giảm `epochs` ở cell 5.

In [ ]:
!python scripts/run_training.py --config config/training.yaml

## 7. Đánh giá TEST trên checkpoint tốt nhất
Training (cell 6) đã đánh giá **val** rồi. Phần này chạy split **test** — giao thức rolling-temporal (graph train+val, mask train+val) mà training **không bao giờ đụng tới**. Đây là số liệu cuối cùng.

(Cell val bên dưới chỉ để xác nhận lại, có thể bỏ qua.)

In [ ]:
import glob, os
cks = sorted(glob.glob('**/best.pt', recursive=True), key=os.path.getmtime)
BEST = cks[-1] if cks else 'checkpoints-final-l4/best.pt'
print('Checkpoint:', BEST)

In [ ]:
# TEST (bắt buộc)
!python scripts/evaluate.py --checkpoint {BEST} --split test

In [ ]:
# VAL (tuỳ chọn — xác nhận lại full-val)
!python scripts/evaluate.py --checkpoint {BEST} --split val

## Ghi chú
- Metric chính: **NDCG@20** (full-ranking trên toàn bộ item).
- Checkpoint & metric lưu cả local (`save_dir`) lẫn W&B artifact trong project `recsys-graph`.
- Nếu A100 hết RAM: giảm `training.batch_size` / `eval_batch_size` ở cell 5.
- Nếu phiên Colab ngắt giữa chừng: chạy lại cell Training — code tự resume từ checkpoint mới nhất trong `save_dir`.